<a href="https://colab.research.google.com/github/Bilal-data/arabic-spam-classifier/blob/main/arabic_spam_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Arabic Spam / Ham Tweet Classifier
TF-IDF + Logistic Regression on **13,240 Arabic tweets** ("Dataset of Arabic Spam and Ham Tweets").

This notebook is the larger-dataset stage of the project. The earlier stage used 50 hand-written messages, where results swung a lot between splits (see the README).

## 1. Load the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

f = "/content/drive/MyDrive/Dataset of Arabic Spam and Ham Tweets (1).xlsx"
tw = pd.read_excel(f)

print(tw.shape)
print(tw.columns.tolist())

(13240, 16)
['Date', 'Time', 'Date Time', 'URL', 'Tweet Text', 'Cleaned Text', 'User Name', 'Location', 'Replied Tweet ID ', 'Replied Tweet User ID', 'Replied Tweet User name', 'Coordinates', 'Retweet Count', 'Favorite Count', 'Favorited', 'Label']


## 2. Check the classes
The data is imbalanced: far more Ham than Spam.

In [ ]:
print(tw["Label"].value_counts())
print(tw[["Tweet Text", "Label"]].isna().sum())

Label
Ham     11299
Spam     1941
Name: count, dtype: int64
Tweet Text    0
Label         0
dtype: int64


## 3. Train / test split
80/20, stratified so both parts keep the same Ham/Spam ratio.

In [ ]:
from sklearn.model_selection import train_test_split

data = tw[["Tweet Text", "Label"]]

X_train, X_test, y_train, y_test = train_test_split(
    data["Tweet Text"], data["Label"],
    test_size=0.2, random_state=42, stratify=data["Label"]
)

print(len(X_train), len(X_test))
print(y_test.value_counts())

10592 2648
Label
Ham     2260
Spam     388
Name: count, dtype: int64


## 4. Model: TF-IDF + Logistic Regression

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipe = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
pipe.fit(X_train, y_train)

print("Accuracy:", pipe.score(X_test, y_test))

Accuracy: 0.9818731117824774


## 5. Precision and recall per class
Accuracy alone is not enough on imbalanced data, so we look at the Spam class separately.

In [ ]:
from sklearn.metrics import classification_report

pred = pipe.predict(X_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

         Ham       0.98      1.00      0.99      2260
        Spam       1.00      0.88      0.93       388

    accuracy                           0.98      2648
   macro avg       0.99      0.94      0.96      2648
weighted avg       0.98      0.98      0.98      2648



## 6. Baseline: what does a model that always says "Ham" score?
This is the number the model must beat for the accuracy to mean anything.

In [ ]:
print("Baseline (always Ham):", round((y_test == "Ham").mean(), 4))

Baseline (always Ham): 0.8535


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, data["Tweet Text"], data["Label"], cv=cv)

print(scores)
print("Mean:", scores.mean())

[0.9826284  0.98074018 0.98036254 0.98187311 0.98225076]
Mean: 0.981570996978852


In [ ]:
from sklearn.metrics import make_scorer, recall_score

spam_recall = make_scorer(recall_score, pos_label="Spam")
r = cross_val_score(pipe, data["Tweet Text"], data["Label"], cv=cv, scoring=spam_recall)

print(r)
print("Mean:", r.mean())

[0.8814433  0.8685567  0.86597938 0.87628866 0.87917738]
Mean: 0.8742890838258288


## Cross-validation results (5-fold, stratified)
- Accuracy: 0.9804 to 0.9826 across folds, mean 0.9816 (very stable)
- Spam recall: 0.866 to 0.881 across folds, mean 0.874 (about 1 in 8 spam tweets is missed)

## Limitations
- Main result uses one 80/20 split; the 5-fold CV above shows it is stable
- Tweets, not SMS or email
- Imbalanced classes; some spam is still missed (recall about 0.87)
- No hyperparameter tuning
- Not production-ready

In [ ]:
dup = tw["Tweet Text"].duplicated(keep=False)

print("extra copies:", tw["Tweet Text"].duplicated().sum())
print("tweets that have an exact copy:", dup.sum())
print()
print(tw.loc[dup, "Label"].value_counts())

extra copies: 260
tweets that have an exact copy: 481

Label
Ham    481
Name: count, dtype: int64


In [ ]:
clean = tw.drop_duplicates(subset="Tweet Text")
print(len(tw), "->", len(clean))

scores2 = cross_val_score(pipe, clean["Tweet Text"], clean["Label"], cv=cv)
r2 = cross_val_score(pipe, clean["Tweet Text"], clean["Label"], cv=cv, scoring=spam_recall)

print("accuracy:", scores2.mean())
print("spam recall:", r2.mean())

13240 -> 12980
accuracy: 0.9813559322033898
spam recall: 0.8753080857604749


In [ ]:
missed = X_test[(y_test == "Spam") & (pred == "Ham")]
caught = X_test[(y_test == "Spam") & (pred == "Spam")]

print("missed:", len(missed), "| caught:", len(caught))
print("avg length missed:", round(missed.str.len().mean(), 1))
print("avg length caught:", round(caught.str.len().mean(), 1))

missed: 48 | caught: 340
avg length missed: 113.3
avg length caught: 105.4


## Error analysis
- On the first 80/20 split, 48 of 388 spam tweets were missed (12%). Missed spam is not shorter than caught spam (about 113 vs 105 characters).
- Reading 20 missed tweets by hand (not shown here, to avoid republishing real users' tweets): most are adult content or ads for products and services, written like normal conversational Arabic with no classic ad phrases. This is a small manual sample, so it is a hypothesis, not a proven cause.
- Exact duplicates: 260 extra copies (481 tweets involved), all labeled ham. Removing them changes nothing (accuracy 98.14%, spam recall 87.5%). Near-duplicates were not checked. Update: a rough check (grouped by the first 40 characters) is in the next section.

In [ ]:
from sklearn.metrics import precision_score

pipe_bal = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000, class_weight="balanced"))
spam_prec = make_scorer(precision_score, pos_label="Spam")

X, y = data["Tweet Text"], data["Label"]

for name, m in [("default", pipe), ("balanced", pipe_bal)]:
    a = cross_val_score(m, X, y, cv=cv).mean()
    r = cross_val_score(m, X, y, cv=cv, scoring=spam_recall).mean()
    p = cross_val_score(m, X, y, cv=cv, scoring=spam_prec).mean()
    print(name, "| acc:", round(a, 4), "| spam recall:", round(r, 4), "| spam precision:", round(p, 4))

default | acc: 0.9816 | spam recall: 0.8743 | spam precision: 1.0
balanced | acc: 0.9957 | spam recall: 0.9737 | spam precision: 0.9968


In [ ]:
import re

def key(t):
    t = re.sub(r"[^\w]", "", str(t))
    return t[:40]

groups = data["Tweet Text"].map(key)
big = groups.value_counts()
shared = groups.isin(big[big > 1].index)

print("unique keys:", groups.nunique(), "of", len(groups))
print("keys with 2+ tweets:", (big > 1).sum())
print("tweets inside those keys:", shared.sum())
print()
print(data.loc[shared, "Label"].value_counts())

unique keys: 11861 of 13240
keys with 2+ tweets: 1019
tweets inside those keys: 2398

Label
Ham     1882
Spam     516
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

gcv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for name, m in [("default", pipe), ("balanced", pipe_bal)]:
    a = cross_val_score(m, X, y, cv=gcv, groups=groups).mean()
    r = cross_val_score(m, X, y, cv=gcv, groups=groups, scoring=spam_recall).mean()
    p = cross_val_score(m, X, y, cv=gcv, groups=groups, scoring=spam_prec).mean()
    print(name, "| acc:", round(a, 4), "| spam recall:", round(r, 4), "| spam precision:", round(p, 4))

default | acc: 0.9808 | spam recall: 0.8692 | spam precision: 1.0
balanced | acc: 0.9944 | spam recall: 0.9649 | spam precision: 0.9968


## Class weights and near-duplicates
- `class_weight="balanced"` (5-fold CV): accuracy 0.9957, spam recall 0.9737, spam precision 0.9968 (default: 0.9816 / 0.8743 / 1.0)
- Grouped CV (tweets with the same first 40 characters stay in one fold): balanced 0.9944 / 0.9649 / 0.9968; default 0.9808 / 0.8692 / 1.0
- About 18% of tweets share their first 40 characters with another tweet (27% of spam, 17% of ham), so near-duplicates exist. The gain from balanced weights survives the grouped check, but only two configurations were compared.

In [17]:
pipe_bi = make_pipeline(
      TfidfVectorizer(ngram_range=(1, 2)),
          LogisticRegression(max_iter=1000, class_weight="balanced"),
)
r = cross_val_score(pipe_bi, X, y, cv=cv, scoring=spam_recall).mean()
print("bigram recall:", round(r, 4))

bigram recall: 0.967


## N-grams
- `ngram_range=(1, 2)` with `class_weight="balanced"` (5-fold CV): spam recall 0.967, vs 0.9737 with single words.
- No improvement: the 0.7-point difference is smaller than the fold-to-fold spread seen earlier (about 1.5 points), so it may be noise. Only recall was measured for this variant; accuracy and precision were not checked.